# Day 042 · mplfinance 画 K 线
**mplfinance** · 阶段 P2 · Python 量化工具栈

> 上一节的 Plotly 强在交互,但画一张正经的 K 线还得自己叠均线、拼成交量。这一节请出一个专门为 K 线而生的工具 mplfinance:它是 matplotlib 的“K 线特供版”,一行 type='candle' 就出红涨绿跌的专业蜡烛图,mav 参数自动叠均线,volume=True 自动加成交量副图,addplot 能把布林带、买卖信号叠上去,style 还能一键换主题。我们用真实 A 股比亚迪,把这些一次打通。

---

**课件生成日期:** 2026-06-10  ·  **建议学习时长:** 16 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [1]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["baostock", "matplotlib", "mplfinance", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


✓ 所有 10 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪 — 现在可以跑下面的代码单元格


## 学习目标

- 知道 mplfinance 是专为 OHLC 蜡烛图设计的工具,数据列名必须是 Open/High/Low/Close/Volume
- 会用一行 mpf.plot(df, type='candle') 画出专业 K 线,对比 matplotlib 手画省多少事
- 会用 mav 参数自动叠多条均线、volume=True 自动加成交量副图
- 会用 make_addplot 把自己算的指标(布林带)和买卖信号点叠到 K 线上
- 会用 style 一键切换图表主题,会用 savefig 调高 dpi 导出清晰图片

## 历史背景:小李为了一张 K 线图,手画到凌晨两点还画歪了

小李刚学会用 matplotlib,想给自己的复盘笔记配一张漂亮的 K 线。他以为很简单,结果发现一根蜡烛要表达开盘、收盘、最高、最低四个价格:实体是开盘到收盘那段矩形,上下还各有一根细影线表示当天最高最低,涨了要红、跌了要绿……他对着文档写了一百多行,循环里一根根画,坐标算错还把实体画到影线外面去,折腾到凌晨两点图还是歪的。第2 天同事看了一眼说:“你直接用 mplfinance 啊,一行 type='candle' 全给你画好,还自动叠均线加成交量。”小李照着写了一行,3 秒出图,专业得像行情软件截下来的。他当场删掉了昨晚那一百多行——这就是专门工具的意义:别人把脏活累活封装好了,你只管调一行。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. mplfinance:K 线特供版的 matplotlib

mplfinance 是 matplotlib 团队专门为金融 K 线做的封装。普通 matplotlib 画蜡烛图要自己算实体和上下影线、判断红涨绿跌,几十上百行;mplfinance 把这套全封装好了,你只要给它一张列名为 Open/High/Low/Close/Volume、索引是日期的表,一行 mpf.plot(df, type='candle') 就出专业 K 线。type 还能换成 'ohlc'(美式条形)、'line'(折线)、'renko'(砖型图)等。


### 2. mav:一个参数自动叠均线

均线是 K 线图的标配。mplfinance 不用你自己 rolling 再 plot,直接给 mav 参数传一组窗口,比如 mav=(5, 20, 60),它就自动算好 5 日、20 日、60 日均线并叠在 K 线上、配好颜色和图例。想看几条均线就写几个数字,极省事。


### 3. volume=True:自动加成交量副图

看 K 线离不开成交量。普通 matplotlib 要自己 make_subplots 再画1 块量副图、还要对齐时间轴;mplfinance 只要 volume=True,自动在 K 线下方加1 块成交量柱状副图,时间轴自动对齐,涨绿跌红也自动配。量价关系一张图说清。


### 4. addplot:把自己的指标叠上去

内置的均线、成交量不够用时,用 mpf.make_addplot 把任何自己算的序列叠到图上。比如布林带(中轨 ± 2 倍标准差3 条线)、自己算的买卖信号点。普通线用默认,信号点用 type='scatter' 配 marker(^ 买、v 卖)。把 make_addplot 做好的列表传给 mpf.plot 的 addplot 参数即可。


### 5. style 换主题 + savefig 高清导出

mplfinance 内置一堆配色主题:charles、yahoo、binance、nightclouds 等,style='yahoo' 一句换一套风格,不用自己调颜色。导出图片用 savefig=dict(fname='k.png', dpi=150),dpi 调高图更清晰,适合放进课件和报告。


## 实操:mplfinance — 一行 type='candle' / mav 自动均线 / volume 副图 / addplot 叠布林带与买卖点 / style 换主题

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [2]:
# day_042_mplfinance.py — mplfinance:专为 K 线而生 / 一行 type='candle' / mav 自动均线 / volume 副图 / addplot 叠指标 / style 换主题
# 用真实 A 股(比亚迪)一行画出专业蜡烛图,自动叠均线、成交量副图,再叠布林带和买卖信号
# 数据:baostock 日线(免费、稳定、国内零翻墙)
# 依赖:pip install mplfinance   (它是 matplotlib 的封装,出图稳)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplfinance as mpf
import baostock as bs

pd.set_option('display.width', 140)
CODE, NAME = 'sz.002594', '比亚迪'
START, END = '2023-01-01', '2024-12-31'

print('==== 0. 数据拉取(baostock 日线 开高低收 + 成交量)====')
lg = bs.login()
if lg.error_code != '0':
    raise RuntimeError(f'baostock 登录失败:{lg.error_msg}')
rs = bs.query_history_k_data_plus(CODE, 'date,open,high,low,close,volume',
                                  start_date=START, end_date=END, frequency='d', adjustflag='2')
rows = []
while rs.error_code == '0' and rs.next():
    rows.append(rs.get_row_data())
bs.logout()
df = pd.DataFrame(rows, columns=['date', 'Open', 'High', 'Low', 'Close', 'Volume'])
df['date'] = pd.to_datetime(df['date'])
for c in ['Open', 'High', 'Low', 'Close', 'Volume']:
    df[c] = df[c].astype(float)
df = df.set_index('date').sort_index()
print(f'{NAME} 日线 {len(df)} 条  {df.index[0].date()} → {df.index[-1].date()}')
print('mplfinance 要求列名正好是 Open/High/Low/Close/Volume,且索引是日期:')
print(df.tail(3).round(2))

# ====================================================================
# mplfinance 内置 style 会覆盖字体设置,基于内置主题套一层中文字体,避免标题/坐标轴变成方块(铁律 33)
CJK_RC = {'font.sans-serif': ['Noto Sans CJK JP', 'WenQuanYi Zen Hei', 'DejaVu Sans'],
          'font.family': 'sans-serif', 'axes.unicode_minus': False}
CJK_CHARLES = mpf.make_mpf_style(base_mpf_style='charles', rc=CJK_RC)
CJK_YAHOO = mpf.make_mpf_style(base_mpf_style='yahoo', rc=CJK_RC)

print('\n==== 1. 一行出专业 K 线 + 均线 + 成交量副图 ====')
# type='candle' 蜡烛图;mav=(5,20,60) 自动叠三条均线;volume=True 自动加成交量副图
mpf.plot(df, type='candle', mav=(5, 20, 60), volume=True, style=CJK_CHARLES,
         title=f'{NAME} K线 + 5/20/60 日均线 + 成交量',
         ylabel='价格(元)', ylabel_lower='成交量',
         savefig=dict(fname='chart_01.png', dpi=130))
print("mpf.plot(df, type='candle', mav=(5,20,60), volume=True):一行搞定 K 线 + 三条均线 + 成交量副图")
print('对比 matplotlib 手画蜡烛图要自己算实体和上下影线、红涨绿跌,这里全自动')

# ====================================================================
print('\n==== 2. addplot 叠加自定义指标:布林带(中轨 ± 2 倍标准差)====')
ma20 = df['Close'].rolling(20).mean()
std20 = df['Close'].rolling(20).std()
boll_up = ma20 + 2 * std20
boll_dn = ma20 - 2 * std20
add_boll = [
    mpf.make_addplot(boll_up, color='#bf616a', width=1.0),
    mpf.make_addplot(ma20, color='#5e81ac', width=1.0),
    mpf.make_addplot(boll_dn, color='#a3be8c', width=1.0),
]
mpf.plot(df, type='candle', volume=True, style=CJK_CHARLES, addplot=add_boll,
         title=f'{NAME} K线 + 布林带(addplot 叠加)',
         ylabel='价格(元)', ylabel_lower='成交量',
         savefig=dict(fname='chart_02.png', dpi=130))
print('make_addplot 把自己算的指标(布林上轨/中轨/下轨)叠到 K 线上')
print('布林带把价格夹在上下轨之间,价格贴上轨偏热、贴下轨偏冷,是常见的波动通道')

# ====================================================================
print('\n==== 3. addplot 标买卖点:5 日线上穿 20 日线买,下穿卖 ====')
ma5 = df['Close'].rolling(5).mean()
cross = (ma5 > ma20).astype(int).diff()
buy = df['Close'].where(cross == 1)    # 金叉日标在收盘价位置,其余为 NaN
sell = df['Close'].where(cross == -1)  # 死叉日
n_buy, n_sell = int((cross == 1).sum()), int((cross == -1).sum())
add_sig = [
    mpf.make_addplot(buy, type='scatter', marker='^', markersize=80, color='#bf616a'),
    mpf.make_addplot(sell, type='scatter', marker='v', markersize=80, color='#3b8a5a'),
    mpf.make_addplot(ma5, color='#d08770', width=0.9),
    mpf.make_addplot(ma20, color='#5e81ac', width=0.9),
]
mpf.plot(df, type='candle', volume=True, style=CJK_YAHOO, addplot=add_sig,
         title=f'{NAME} K线 + 金叉买点/死叉卖点(style=yahoo 换主题)',
         ylabel='价格(元)', ylabel_lower='成交量',
         savefig=dict(fname='chart_03.png', dpi=130))
print(f"type='scatter' 的 addplot 把信号点标在 K 线上:金叉 {n_buy} 个买点(红三角),死叉 {n_sell} 个卖点(绿三角)")
print("style=CJK_YAHOO 一键换成另一套配色主题,charles/yahoo/binance 等内置主题随便切")

# 小结数字(供文案核对)
period_ret = df['Close'].iloc[-1] / df['Close'].iloc[0] - 1
print(f'\n[小结] {NAME} 区间 {df.index[0].date()}→{df.index[-1].date()} 涨跌幅 {period_ret*100:.1f}%')
print(f'最高 {df["High"].max():.2f} 元 / 最低 {df["Low"].min():.2f} 元 / 期末 {df["Close"].iloc[-1]:.2f} 元')
print(f'金叉 {n_buy} 次 / 死叉 {n_sell} 次')
print('\n[done] 3 张专业 K 线图已生成(K线+均线+量 / 布林带 / 买卖信号)')

==== 0. 数据拉取(baostock 日线 开高低收 + 成交量)====
login success!
logout success!
比亚迪 日线 484 条  2023-01-03 → 2024-12-31
mplfinance 要求列名正好是 Open/High/Low/Close/Volume,且索引是日期:
             Open   High    Low  Close     Volume
date                                             
2024-12-27  94.47  94.72  93.22  94.30  7868319.0
2024-12-30  94.61  95.36  93.33  93.88  8294681.0
2024-12-31  93.88  94.38  92.89  93.11  7761976.0

==== 1. 一行出专业 K 线 + 均线 + 成交量副图 ====
mpf.plot(df, type='candle', mav=(5,20,60), volume=True):一行搞定 K 线 + 三条均线 + 成交量副图
对比 matplotlib 手画蜡烛图要自己算实体和上下影线、红涨绿跌,这里全自动

==== 2. addplot 叠加自定义指标:布林带(中轨 ± 2 倍标准差)====
make_addplot 把自己算的指标(布林上轨/中轨/下轨)叠到 K 线上
布林带把价格夹在上下轨之间,价格贴上轨偏热、贴下轨偏冷,是常见的波动通道

==== 3. addplot 标买卖点:5 日线上穿 20 日线买,下穿卖 ====
type='scatter' 的 addplot 把信号点标在 K 线上:金叉 21 个买点(红三角),死叉 20 个卖点(绿三角)
style=CJK_YAHOO 一键换成另一套配色主题,charles/yahoo/binance 等内置主题随便切

[小结] 比亚迪 区间 2023-01-03→2024-12-31 涨跌幅 10.3%
最高 111.35 元 / 最低 52.97 元 / 期末 93.11 元
金叉 21 次 / 死叉 20 次

[done] 3 张专业 K 线图已生成(K线+均线+量 / 

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| A 股 K 线 | sz.002594 | 比亚迪日线一行画出 K 线 + 5/20/60 日均线 + 成交量副图,专业得像行情软件 |
| 技术指标 |  | 用 addplot 把布林带(中轨 ± 2 倍标准差)叠到 K 线上,看价格在通道里的位置 |
| 信号回放 |  | 5 日线上穿 20 日线标红三角买点、下穿标绿三角卖点,金叉死叉一眼回放 |
| 复盘配图 |  | 复盘笔记/公众号配图用 style 换主题 + 高 dpi 导出,几秒出一张专业 K 线 |


## 常见坑

### ⚠ 01. 列名不对直接报错

mplfinance 死磕列名:必须正好是 Open、High、Low、Close、Volume(首字母大写),且索引是 DatetimeIndex。从 baostock 拉回来的是小写 open/high/low/close,必须先改名,否则一画就报错。这是新手最常踩的第一个坑。

### ⚠ 02. 索引不是日期类型

如果 date 列没转成 pd.to_datetime 再 set_index,索引是字符串,mplfinance 不认,K 线横轴会乱或报错。拉完数据先 to_datetime + set_index + sort_index 三件套。

### ⚠ 03. addplot 序列长度和 df 对不齐

make_addplot 的序列必须和 df 一样长、索引一一对应。想只在某些天标信号点,其余位置要填 NaN(用 where 把非信号日变 NaN),而不是只传几个点,否则会错位或报错。

### ⚠ 04. 用 returnfig 还是 savefig 搞混

想直接存图用 savefig=dict(fname=..., dpi=...);想拿到 fig/axes 再加工要 returnfig=True 取回 (fig, axes) 自己 savefig。两种别混用,否则要么没存上、要么拿不到对象。

### ⚠ 05. 数据量太大图挤成一团

几千根 K 线挤在一张图里,蜡烛细成线、看不清。画长周期时要么只取最近一段,要么换成 type='line' 看趋势,K 线适合看几个月到一两年的中等长度。

## 实战 SOP · mplfinance K 线作图实战 6 条 SOP

1. 先对齐列名和索引 — 改成 Open/High/Low/Close/Volume + DatetimeIndex,再画。
2. 一行出 K 线 — mpf.plot(df, type='candle'),需要均线加 mav,需要量加 volume=True。
3. 自定义指标用 addplot — make_addplot 叠布林带等,先做成 list 再传。
4. 信号点用 type='scatter' + where 填 NaN — 只在信号日有点,其余留空。
5. 换主题用 style,导出用 savefig + 高 dpi — 一句换风格,dpi 调高更清晰。
6. K 线只画中等长度 — 几个月到一两年最清楚,太长换折线看趋势。

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. mplfinance 是 matplotlib 的 K 线特供版,一行 type='candle' 出专业蜡烛图,告别手画上百行。
3. 数据列名必须正好是 Open/High/Low/Close/Volume,索引是日期,这是第一道门槛。
4. mav 参数自动叠多条均线,volume=True 自动加成交量副图、时间轴自动对齐。
5. make_addplot 把布林带、买卖信号等自己算的东西叠到 K 线上,信号点用 type='scatter'。
6. style 一键换配色主题(charles/yahoo/binance),savefig + 高 dpi 导出清晰图。
7. addplot 的序列要和 df 一样长、对齐索引,非信号日填 NaN 防错位。

## 自测题

**Q1.** 用 mplfinance 画 K 线前,数据要满足什么条件?(提示:列名必须正好是 Open/High/Low/Close/Volume,首字母大写;索引必须是 DatetimeIndex 日期类型。baostock 拉回来是小写列名,要先改名 + to_datetime + set_index。)

**Q2.** 怎么让 K 线图自动叠均线和成交量副图?(提示:mpf.plot 里加 mav=(5,20,60) 自动叠3 条均线,加 volume=True 自动在下方加1 块成交量副图、时间轴自动对齐,都不用自己 rolling 和 make_subplots。)

**Q3.** 想把自己算的布林带或买卖点叠到 K 线上,用什么?(提示:用 mpf.make_addplot 把序列做成附加图层,普通线直接传,信号点用 type='scatter' 配 marker;把多个 make_addplot 放进 list 传给 mpf.plot 的 addplot 参数。)

**Q4.** 只想在金叉那几天标买点,序列该怎么准备?(提示:做一个和 df 一样长的序列,只有金叉日是收盘价、其余位置是 NaN,可以用 df['Close'].where(条件) 实现;mplfinance 会跳过 NaN 只在有值的天画点。)

把答案写下来,3 天后再回看。

## 下一节预告

**Day 043 · yfinance 拉数据** (yfinance)

前面我们都在用 baostock 的国内数据,下一节换条路去拉海外数据:用 yfinance 一行就能把美股的开高低收拉回来,还能批量拉一篮子股票5 年的行情,看看免费拿全球数据有多方便。

## 推荐阅读

- mplfinance 官方文档《Plotting Basics》(2024)— type/mav/volume/style 参数的权威用法
- mplfinance 官方文档《Add Your Own Plots (addplot)》(2024)— make_addplot 叠加指标与信号的标准范式
- John Murphy《Technical Analysis of the Financial Markets》(1999, NYIF)— K 线、均线、成交量等技术分析图表的含义
- Steve Nison《Japanese Candlestick Charting Techniques》(2001, Prentice Hall)— 蜡烛图的起源与形态解读经典
- Wes McKinney《Python for Data Analysis》(2022, O'Reilly)— pandas 时间序列整理,为作图准备数据